In [1]:
import numpy as np
import matplotlib.pyplot as plt

# Main system parameters: number of mics, number of samples in time
Nmics = 64
Nsamp = 200

# Source: x, y coordinates: x: 0+, y: [-Y, +Y] where Y determined by pitch and Nmics
src = (0, 0)
# Spacing between microphones
pitch = 0.1
# Proxy for sampling rate
dist_per_samp = 0.1
# Speed of sound in the medium
C = 2
# Time dilation factor for sinc pulse: how narrow
SincP = 5.0
# CODE Locations of microphones
mics = [(0, pitch * (i - Nmics // 2)) for i in range(Nmics)]

In [2]:
def wsrc(t):
    return np.sinc(SincP * t)

## Question 1

To create the plots with varying sinc pulse widths, you can manipulate the `SincP` parameter within the `wsrc` function. This parameter is responsible for determining the width of the sinc pulse. If you increase the value of `SincP`, it will make the sinc pulse narrower, while decreasing the value will result in a wider sinc pulse. Adjusting this parameter allows you to control the width of the sinc pulses in your generated plots..



In [ ]:
mic_outputs = np.zeros((Nmics, Nsamp))

# Time axis for the wave
t = np.arange(0, Nsamp * dist_per_samp, dist_per_samp)

# Create the source signal (sinc pulse)
source_signal = wsrc(t)
plt.plot(t,source_signal)
plt.savefig('Part_1.png')
plt.show()


In [ ]:
#Generate samples corresponding to each mic for a given obstacle position and parameters
def sample_generator(obstacle, Nmics, Nsamp, pitch, dist_per_samp, C, SincP):
    t_axis = np.linspace(0, dist_per_samp * Nsamp, Nsamp)
    mic_outputs = np.zeros((Nmics, Nsamp))
    mic_outputs1 = np.zeros((Nmics, Nsamp))

    for i in range(Nmics):
        for j in range(Nsamp):
            pt = obstacle
            d = dist(src, pt, mics[i])
            delay = d / C
            mic_outputs1[i][j] = wsrc(t_axis[j] - delay)+mics[i][1]
            mic_outputs[i][j] = wsrc(t_axis[j] - delay)

    return mic_outputs,mic_outputs1# Function that computes the sound wave from the source at a specific time.


# Function to calculate the distance from src to a mic after reflecting through pt
def dist(src, pt, mic):
    d1 = np.sqrt((pt[0] - src[0]) ** 2 + (pt[1] - src[1]) ** 2)
    d2 = np.sqrt((mic[0] - pt[0]) ** 2 + (mic[1] - pt[1]) ** 2)
    return d1 + d2

In [ ]:
#Generate samples corresponding to each mic for a given obstacle position and parameters
def sample_generator(obstacle, Nmics, Nsamp, pitch, dist_per_samp, C, SincP):
    t_axis = np.linspace(0, dist_per_samp * Nsamp, Nsamp)
    mic_outputs = np.zeros((Nmics, Nsamp))
    mic_outputs1 = np.zeros((Nmics, Nsamp))

    for i in range(Nmics):
        for j in range(Nsamp):
            pt = obstacle
            d = dist(src, pt, mics[i])
            delay = d / C
            mic_outputs1[i][j] = wsrc(t_axis[j] - delay)+mics[i][1]
            mic_outputs[i][j] = wsrc(t_axis[j] - delay)

    return mic_outputs,mic_outputs1

In [ ]:
# Choose an obstacle position dynamically
obstacle = (3, -1)

# Generate samples
samples,samples1 = sample_generator(obstacle, Nmics, Nsamp, pitch, dist_per_samp, C, SincP)

In [ ]:
# # Plot the microphone outputs
for j in range(Nmics):
    plt.plot(samples1[j,:])

plt.title('Outputs')
plt.xlabel('Time')
plt.ylabel('Amplitude')
plt.show()

In [ ]:
# Visualize the microphone outputs
plt.imshow(samples, aspect='auto',cmap='viridis', extent=(0, Nsamp, -pitch * Nmics / 2, pitch * Nmics / 2), origin='lower')
plt.colorbar(label='Amplitude')
plt.title('Outputs')

plt.xlabel('Time')
plt.ylabel('Microphone Locations')
plt.show()

In [ ]:
def reconsrt_n_visualize1(src, mics, samples, Nsamp, dist_per_samp, pitch, C):
    x_axis = np.linspace(0, dist_per_samp * Nsamp, Nsamp)
    y_axis = [pitch * (i - len(mics) // 2) for i in range(len(mics))]
    reconstructed = np.zeros((len(mics),Nsamp))

    for x in range(len(y_axis)):
        for y in range(len(x_axis)):
            pt1 = (x_axis[y], y_axis[x])
            for i in range(len(mics)):
                delay = dist(src, pt1, mics[i]) / C
                nearest_index = np.argmin(np.abs(x_axis - delay))
                reconstructed[x, y] += samples[i][nearest_index]
    plt.imshow(reconstructed, aspect='auto', cmap='viridis')
    plt.colorbar(label='Amplitude')
    plt.title('Microphone Outputs')
    plt.xlabel('Time')
    plt.ylabel('Microphone Locations')
    plt.savefig('Part_2.png')
    plt.show()
    
    # Answering Questions:
    # QUESTION 1: (x, y) coordinates corresponding to the maximum amplitude (yellow color)
    max_coords = np.unravel_index(reconstructed.argmax(), reconstructed.shape)
    print(f"Coordinates of maximum amplitude: {max_coords}")

    # QUESTION 2: Maximum obstacle x- and y- coordinate for image reconstruction
    max_x_coord = Nsamp * dist_per_samp
    max_y_coord = pitch * Nmics / 2
    print(f"Maximum obstacle x-coordinate: {max_x_coord}, Maximum obstacle y-coordinate: {max_y_coord}")

reconsrt_n_visualize1(src, mics, samples, Nsamp, dist_per_samp, pitch, C)


In [ ]:
def reconsrt_n_visualize2(src, mics, samples, Nsamp, dist_per_samp, pitch, C):
    x_axis = np.linspace(0, dist_per_samp * Nsamp, Nsamp)
    y_axis = [pitch * (i - len(mics) // 2) for i in range(len(mics))]
    reconstructed = np.zeros((len(mics), Nsamp))
    for x in range(len(mics)):
        for y in range(Nsamp):
            pt1 = (x_axis[y], y_axis[x])
            for j in range(len(mics)):
                delay = dist(src, pt1, mics[j])
                if  0<= delay < Nsamp*dist_per_samp:
                    reconstructed[x,y] += samples[j][int(delay/dist_per_samp)]

    # reconstructed = reconstructed.T

    plt.imshow(reconstructed, aspect='auto',cmap='viridis')
    plt.colorbar(label='Amplitude')
    plt.title('Microphone Outputs')
    plt.xlabel('Time')
    plt.ylabel('Microphone Locations')
    plt.show()
    # reconstructed = reconstructed.T

    # Answering Questions:
    # QUESTION 1: (x, y) coordinates corresponding to the maximum amplitude (yellow color)
    max_coords = np.unravel_index(reconstructed.argmax(), reconstructed.shape)
    print(f"Coordinates of maximum amplitude: {max_coords}")

    # QUESTION 2: Maximum obstacle x- and y- coordinate for image reconstruction
    max_x_coord = Nsamp * dist_per_samp
    max_y_coord = pitch * Nmics / 2
    print(f"Maximum obstacle x-coordinate: {max_x_coord}, Maximum obstacle y-coordinate: {max_y_coord}")

**Explanation of Maximum Amplitude Coordinates:**

The point `(30, 22)`, where the maximum amplitude (depicted in yellow) is observed, signifies the anticipated position of the source or obstacle. This alignment adheres to the principles governing the propagation of acoustic waves, as implemented in the provided code.

### Key Factors for Accurate Positioning:

1. **Time-of-Flight and Delay Calculation**: Within the `generate_samples` function, the delay in wave arrival at each microphone is computed based on the source-to-microphone distance and the speed of sound (`C`). These delays are integrated into the generated signals. The coordinates `(30, 22)` likely represent the location where time-of-flight and delays coincide to produce constructive interference, resulting in the maximum amplitude.

2. **Interference Patterns**: Reflections and interactions of sound waves with the environment give rise to interference patterns. The `(30, 22)` coordinates likely correspond to a point where constructive interference contributes to a peak in amplitude. The relative phases of the waves contribute to this observed maximum amplitude.

3. **Optimal Alignment**: The position `(30, 22)` may signify the optimal alignment for reflected waves from the obstacle to reach the microphones, maximizing amplitude. This alignment is influenced by the specific geometry of the obstacle, microphone placement, and chosen simulation parameters.

4. **Influence of Simulation Parameters**: Parameters like `pitch`, `dist_per_samp`, and `SincP` in the simulation code affect the spatial and temporal characteristics of the generated signals. Adjusting these parameters can further impact the expected position of maximum amplitude.

In summary, the `(30, 22)` coordinates represent the location where simulated acoustic signals, considering delays, reflections, and interference, result in the highest amplitude at the microphones. This alignment is a consequence of the simulation setup and the characteristics of the simulated sound waves interacting with the obstacle and microphone array.


In [ ]:
# Load the additional data files
data_file1 = 'rx2.txt'  # Replace with the actual file path
data_file2 = 'rx3.txt'  # Replace with the actual file path

# Load mic outputs from the data filesimport numpy as np
import matplotlib.pyplot as plt

# Define the main system parameters
src = (0, 0)
pitch = 0.1
dist_per_samp = 0.1
C = 2
SincP = 5.0

# Function to calculate the source sound wave at a given time
def wsrc(t):
    return np.sinc(SincP * t)

# Function to calculate the distance from src to a mic after reflecting through pt
def dist(src, pt, mic):
    d1 = np.sqrt((pt[0] - src[0]) ** 2 + (pt[1] - src[1]) ** 2)
    d2 = np.sqrt((mic[0] - pt[0]) ** 2 + (mic[1] - pt[1]) ** 2)
    return d1 + d2

mic_outputs1 = np.loadtxt(data_file1)
mic_outputs2 = np.loadtxt(data_file2)

In [ ]:
reconsrt_n_visualize2(src, mics, mic_outputs1, Nsamp=200, dist_per_samp=0.1, pitch=0.1, C=0.5)
reconsrt_n_visualize2(src, mics, mic_outputs2, Nsamp=200, dist_per_samp=0.1, pitch=0.1, C=0.5)

## Influence of Speed of Sound (\(C\)) on Image Definition

A decrease in the speed of sound (\(C\)) within the simulation has a noticeable effect on the sharpness of the reconstructed image. This change is rooted in the principles of sound wave propagation and the pivotal role of \(C\) in determining the time delays associated with reflected waves.

### Simplified Explanation:

1. **Time Delays and Wavefronts:**
   - The time delay (\(delay\)) introduced in the simulation is calculated as the distance divided by the speed of sound (\(d/C\)). A reduction in \(C\) results in an increased time delay for a given distance.
   - Slower sound propagation implies that the reflected wavefronts take longer to reach the microphones.

2. **Spatial Resolution:**
   - In image reconstruction, the timing of received signals holds significance.
   - A slower speed of sound widens the temporal gap between signals received by different microphones for a given reflected wave.
   - This heightened temporal gap contributes to enhanced temporal resolution during image reconstruction.

3. **Sharper Image:**
   - Improved temporal resolution facilitates clearer differentiation between signals arriving at different times, leading to a sharper image.
   - Increased time delays contribute to a more distinct representation of reflections and interactions with the environment.

4. **Effect on Interference Patterns:**
   - The interference patterns in the simulation are influenced by the relative phases of waves reaching the microphones.
   - Changes in \(C\) impact these phase relationships, and a slower speed may intensify interference patterns.

### Summary:

Reducing the speed of sound (\(C\)) in the simulation captures finer temporal details in received signals, resulting in a sharper image. This outcome is attributed to increased time delays and improved temporal resolution in depicting reflected waves.

Feel free to experiment with different values of \(C\) to observe how they affect the clarity of the reconstructed images in your simulation.


In [ ]:
# Function to generate samples for a given obstacle position and parameters
def sample_generate(obstacle, Nmics, Nsamp, pitch, dist_per_samp, C, SincP):
    t_axis = np.linspace(0, dist_per_samp * Nsamp, Nsamp)
    mic_outputs = np.zeros((Nmics, Nsamp))

    for i in range(Nmics):
        for j in range(Nsamp):
            pt = obstacle
            d = dist(src, pt, mics[i])
            delay = d / C
            mic_outputs[i][j] = wsrc(t_axis[j] - delay)

    return mic_outputs

# Function to reconstruct and visualize the image
def reconstruct_and_visualize(Nmics, Nsamp):
    mics = [(0, pitch * (i - Nmics // 2)) for i in range(Nmics)]
    obstacle = (3, -1)

    # Generate samples
    samples = sample_generate(obstacle, Nmics, Nsamp, pitch, dist_per_samp, C, SincP)

    # Reconstruct and visualize the image
    x_axis = np.linspace(0, dist_per_samp * Nsamp, Nsamp)
    y_axis = [pitch * (i - len(mics) // 2) for i in range(len(mics))]
    reconstructed = np.zeros((len(mics),Nsamp))

    for x in range(len(y_axis)):
        for y in range(len(x_axis)):
            pt1 = (x_axis[y], y_axis[x])
            for i in range(len(mics)):
                delay = dist(src, pt1, mics[i]) / C
                nearest_index = np.argmin(np.abs(x_axis - delay))
                reconstructed[x, y] += samples[i][nearest_index]

    # Visualize the reconstructed image
    plt.imshow(reconstructed, aspect='auto', cmap='viridis')
    plt.colorbar(label='Amplitude')
    plt.title(f'Reconstructed Image (Nmics={Nmics}, Nsamp={Nsamp})')
    plt.xlabel('Time')
    plt.ylabel('Microphone Locations')
    plt.show()

# Perform experiments with different combinations of Nmics and Nsamp
for Nmics in [8, 32, 64]:
    for Nsamp in [50, 100, 200]:
        reconstruct_and_visualize(Nmics, Nsamp)
